In [3]:
import torch
import torch.nn as nn
import math

In [2]:
!pip install torch

  Using cached torch-2.11.0-cp314-cp314-win_amd64.whl.metadata (29 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.11.0-cp314-cp314-win_amd64.whl (114.6 MB)
Using cached fsspec-2026.3.0-py3-none-any.whl (202 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached filelock-3.25.2-py3-none-any.whl (26 kB)

   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        
        self.d_model = d_model
        self.heads = heads
        self.head_dim = d_model // heads
        
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        
        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, x):
        N, seq_len, d_model = x.shape
        
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)
        
        Q = Q.view(N, seq_len, self.heads, self.head_dim)
        K = K.view(N, seq_len, self.heads, self.head_dim)
        V = V.view(N, seq_len, self.heads, self.head_dim)
        
        energy = torch.einsum("nqhd,nkhd->nhqk", Q, K)
        attention = torch.softmax(energy / math.sqrt(self.head_dim), dim=-1)
        
        out = torch.einsum("nhql,nlhd->nqhd", attention, V)
        out = out.reshape(N, seq_len, self.d_model)
        
        return self.fc_out(out)

In [6]:
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, d_model)
        )

    def forward(self, x):
        return self.net(x)

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, heads, hidden_dim, dropout=0.1):
        super().__init__()
        
        self.attention = MultiHeadAttention(d_model, heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.ff = FeedForward(d_model, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn = self.attention(x)
        x = self.norm1(x + self.dropout(attn))
        
        ff = self.ff(x)
        x = self.norm2(x + self.dropout(ff))
        
        return x

In [8]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, heads, num_layers, hidden_dim, max_len, num_classes):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, heads, hidden_dim)
            for _ in range(num_layers)
        ])
        
        self.fc_out = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        
        for layer in self.layers:
            x = layer(x)
        
        x = x.mean(dim=1)  # pooling
        
        return self.fc_out(x)